In [124]:
import pandas as pd
import json

# Caminho para seu arquivo
file_path = 'similarity_results.jsonl'

# Carrega cada linha do arquivo como um dicionário
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        # Extrai campos desejados
        row = {
            'similarity_result': item['similarity_result'],
            'diagnosed_name': item['diagnosed']['name'],
            'diagnosed_given': item['diagnosed']['given'],
            'diagnosed_then': item['diagnosed']['then'],
            'candidate_name': item['candidate']['name'],
            'candidate_given': item['candidate']['given'],
            'candidate_then': item['candidate']['then'],
            'elapsed_time': item['elapsed_time'],
            'config': item['config'],
        }
        data.append(row)

# Cria o DataFrame
df = pd.DataFrame(data)

# Exemplo: mostra as primeiras linhas
df


,similarity_result,diagnosed_name,diagnosed_given,diagnosed_then,candidate_name,candidate_given,candidate_then,elapsed_time,config
0,0.913015,diagnosed_baseline,h < 100 AND vibration >= 63,delta_dt >= 10 AND h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.074152,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
1,0.836667,diagnosed_baseline,h < 100 AND vibration >= 63,delta_dt >= 10 AND h >= 100,cand_limited_satellite_flying,h > 10 AND vibration >= 30,delta_dt >= 10 AND h >= 100 AND satellite_coun...,0.073341,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
2,0.628622,diagnosed_baseline,h < 100 AND vibration >= 63,delta_dt >= 10 AND h >= 100,cand_hover_mode_flying,h < 70 AND vibration <= 80,delta_dt >= 10 AND h <= 80 AND delta_flight_ti...,0.063126,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
3,0.721500,diagnosed_plus_25,h < 10 AND vibration >= 0,delta_dt >= 10 AND h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.071963,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
4,0.767460,diagnosed_plus_25,h < 10 AND vibration >= 0,delta_dt >= 10 AND h >= 100,cand_limited_satellite_flying,h > 10 AND vibration >= 30,delta_dt >= 10 AND h >= 100 AND satellite_coun...,0.066586,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
5,0.639199,diagnosed_plus_25,h < 10 AND vibration >= 0,delta_dt >= 10 AND h >= 100,cand_hover_mode_flying,h < 70 AND vibration <= 80,delta_dt >= 10 AND h <= 80 AND delta_flight_ti...,0.076540,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
6,0.730760,diagnosed_plus_50,h < 10 AND vibration >= 10,delta_dt >= 10 AND h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.060174,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
7,0.780424,diagnosed_plus_50,h < 10 AND vibration >= 10,delta_dt >= 10 AND h >= 100,cand_limited_satellite_flying,h > 10 AND vibration >= 30,delta_dt >= 10 AND h >= 100 AND satellite_coun...,0.069767,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
8,0.622532,diagnosed_plus_50,h < 10 AND vibration >= 10,delta_dt >= 10 AND h >= 100,cand_hover_mode_flying,h < 70 AND vibration <= 80,delta_dt >= 10 AND h <= 80 AND delta_flight_ti...,0.081029,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
9,0.668795,diagnosed_plus_75,h < 10 AND vibration >= 10,delta_dt >= 10 AND h >= 20,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.056728,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."


In [125]:
# df_baseline = df[df['diagnosed_name'] == 'diag_baseline']
# df_baseline

# import plotly.express as px

# fig = px.scatter(
#     df_baseline,
#     x='candidate_name',           # eixo X: nome do candidato
#     y='similarity_result',        # eixo Y: similaridade
#     color='similarity_result',    # cor dos pontos pela similaridade
#     hover_data=['candidate_given', 'candidate_then'],  # informações extras no hover
#     title='Similaridade por Candidato (Diagnosed Baseline)'
# )
# fig.update_layout(
#     xaxis_title='Candidato',
#     yaxis_title='Similaridade',
#     template='plotly_white'
# )
# fig.show()

In [126]:
# keep_candidates = ["6", "7", "12", "15", "18", "19", "5", "cand_baseline"] # esses sao os que eu mais gostei para visualizar

# df = df[df["candidate_name"].astype(str).isin(keep_candidates)]
# df


In [127]:
# mapping = {
#     "6": 1,
#     "7": 2,
#     "12":  3,
#     "15":  4,
#     "18":  5,
#     "19": 6,
#     "5":  7,
#     # "diagnosed_baseline" não entra no mapping para permanecer igual
# }

# # Garante comparação por string, aplica o mapeamento e preserva o que não estiver no dict
# df['candidate_name'] = (
#     df['candidate_name'].map(mapping)
#                          .fillna(df['candidate_name'])
# )


In [128]:
import pandas as pd
import plotly.graph_objects as go

# Ordem dos diagnósticos no eixo X:
# primeiro os com erro negativo, depois o baseline, depois os positivos (25% em 25%)
diagnosis_order = [
    "diagnosed_minus_100",
    "diagnosed_minus_75",
    "diagnosed_minus_50",
    "diagnosed_minus_25",
    "diagnosed_baseline",
    "diagnosed_plus_25",
    "diagnosed_plus_50",
    "diagnosed_plus_75",
    "diagnosed_plus_100",
]

# rótulos bonitos para o eixo X (em termos de erro percentual)
error_labels = [
    "-100%", "-75%", "-50%", "-25%",
    "0%",    # baseline
    "+25%", "+50%", "+75%", "+100%",
]

# Pivot: linhas = candidatos, colunas = diagnosticados, valores = similaridade
df_pivot = df.pivot(
    index="candidate_name",
    columns="diagnosed_name",
    values="similarity_result"
)

# Garante que só usamos as colunas que realmente existem no df
cols_present = [col for col in diagnosis_order if col in df_pivot.columns]
df_pivot = df_pivot[cols_present]

# Ranking (1 = melhor similaridade)
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()

for cand in df_rank.index:
    similarity_texts = [f"{df_pivot.loc[cand, col]:.2f}" for col in df_pivot.columns]
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=3),
        marker=dict(size=10),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=9)
    ))

# Índice do baseline na ordem efetivamente usada
baseline_name = "diagnosed_baseline"
if baseline_name in df_pivot.columns:
    baseline_idx = df_pivot.columns.tolist().index(baseline_name)

    # Linhas verticais tracejadas finas ao redor do baseline
    fig.add_vline(
        x=baseline_idx - 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )
    fig.add_vline(
        x=baseline_idx + 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )

fig.update_layout(
    title='Ranking dos candidatos sob erros paramétricos (±25%) no diagnóstico',
    yaxis=dict(
        autorange='reversed',
        dtick=1,
        title='Ranking (1 = melhor)',
    ),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=df_pivot.columns.tolist(),
        tickvals=df_pivot.columns.tolist(),
        # usa labels de erro percentual, alinhando pelo mesmo comprimento
        ticktext=error_labels[:len(df_pivot.columns)]
    ),
    width=1000,
    height=700,
    legend=dict(
        title="Candidatos",
        orientation="h",
        xanchor="center", x=0.5,
        yanchor="top",    y=-0.20,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=11)
    ),
    margin=dict(l=40, r=40, t=70, b=160),
    font=dict(
        family="Arial",
        size=12
    )
)

fig.show()


In [129]:
import pandas as pd
import plotly.graph_objects as go

# Ordem dos diagnósticos no eixo X:
diagnosis_order = [
    "diagnosed_minus_100",
    "diagnosed_minus_75",
    "diagnosed_minus_50",
    "diagnosed_minus_25",
    "diagnosed_baseline",
    "diagnosed_plus_25",
    "diagnosed_plus_50",
    "diagnosed_plus_75",
    "diagnosed_plus_100",
]

error_labels = [
    "-100%", "-75%", "-50%", "-25%",
    "0%",
    "+25%", "+50%", "+75%", "+100%",
]

df_pivot = df.pivot(
    index="candidate_name",
    columns="diagnosed_name",
    values="similarity_result"
)

cols_present = [col for col in diagnosis_order if col in df_pivot.columns]
df_pivot = df_pivot[cols_present]

fig = go.Figure()

for cand in df_pivot.index:
    similarity_vals = df_pivot.loc[cand].values
    similarity_texts = [f"{v:.2f}" for v in similarity_vals]

    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=similarity_vals,
        mode='lines+markers+text',   # <- volta com texto
        name=str(cand),
        line=dict(width=3),
        marker=dict(size=8),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=11),      # um pouco menor pra não poluir tanto
    ))

baseline_name = "diagnosed_baseline"
if baseline_name in df_pivot.columns:
    baseline_idx = df_pivot.columns.tolist().index(baseline_name)

    fig.add_vline(
        x=baseline_idx - 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )
    fig.add_vline(
        x=baseline_idx + 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )

fig.update_layout(
    title='Similaridade dos candidatos sob erros paramétricos (±25%) no diagnóstico',
    yaxis=dict(
        title='Similarity',
        range=[0.4, 1],
        dtick=0.05
    ),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=df_pivot.columns.tolist(),
        tickvals=df_pivot.columns.tolist(),
        ticktext=error_labels[:len(df_pivot.columns)]
    ),
    width=600,
    height=350,
    legend=dict(
        title="Candidatos",
        orientation="h",
        xanchor="center", x=0.5,
        yanchor="top",    y=-0.30,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=9)
    ),
    margin=dict(l=60, r=20, t=50, b=110),
    font=dict(
        family="Arial",
        size=11
    )
)
fig.write_image("rq3.2.pdf")
fig.show()





In [130]:
import pandas as pd
import plotly.graph_objects as go

# Ordem dos diagnósticos no eixo X:
diagnosis_order = [
    "diagnosed_minus_100",
    "diagnosed_minus_75",
    "diagnosed_minus_50",
    "diagnosed_minus_25",
    "diagnosed_baseline",
    "diagnosed_plus_25",
    "diagnosed_plus_50",
    "diagnosed_plus_75",
    "diagnosed_plus_100",
]

# Rótulos bonitos para o eixo X (erro percentual)
error_labels = [
    "-100%", "-75%", "-50%", "-25%",
    "0%",
    "+25%", "+50%", "+75%", "+100%",
]

# Pivot: linhas = candidatos, colunas = diagnósticos, valores = similaridade
df_pivot = df.pivot(
    index="candidate_name",
    columns="diagnosed_name",
    values="similarity_result"
)

# Garante que só usamos as colunas na ordem desejada e que existem no df
cols_present = [col for col in diagnosis_order if col in df_pivot.columns]
df_pivot = df_pivot[cols_present]

fig = go.Figure()

# Apenas linhas + marcadores, sem texto
for cand in df_pivot.index:
    similarity_vals = df_pivot.loc[cand].values

    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=similarity_vals,
        mode='lines+markers',   # <- sem 'text'
        name=str(cand),
        line=dict(width=3),
        marker=dict(size=10),
    ))

# Índice do baseline na ordem efetivamente usada
baseline_name = "diagnosed_baseline"
if baseline_name in df_pivot.columns:
    baseline_idx = df_pivot.columns.tolist().index(baseline_name)

    fig.add_vline(
        x=baseline_idx - 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )
    fig.add_vline(
        x=baseline_idx + 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )

fig.update_layout(
    title='Similaridade dos candidatos sob erros paramétricos (±25%) no diagnóstico',
    yaxis=dict(
        title='Similarity',
        range=[0.4, 1],
        dtick=0.05
    ),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=df_pivot.columns.tolist(),
        tickvals=df_pivot.columns.tolist(),
        ticktext=error_labels[:len(df_pivot.columns)]
    ),
    width=1000,
    height=700,
    legend=dict(
        title="Candidatos",
        orientation="h",
        xanchor="center", x=0.5,
        yanchor="top",    y=-0.20,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=11)
    ),
    margin=dict(l=40, r=40, t=70, b=160),
    font=dict(
        family="Arial",
        size=12
    )
)

fig.show()


In [131]:
import pandas as pd
import plotly.graph_objects as go

# Ordem dos diagnósticos no eixo X:
diagnosis_order = [
    "diagnosed_minus_100",
    "diagnosed_minus_75",
    "diagnosed_minus_50",
    "diagnosed_minus_25",
    "diagnosed_baseline",
    "diagnosed_plus_25",
    "diagnosed_plus_50",
    "diagnosed_plus_75",
    "diagnosed_plus_100",
]

# Rótulos bonitos para o eixo X (erro percentual)
error_labels = [
    "-100%", "-75%", "-50%", "-25%",
    "0%",
    "+25%", "+50%", "+75%", "+100%",
]

# Pivot: linhas = candidatos, colunas = diagnosticados, valores = similaridade
df_pivot = df.pivot(
    index="candidate_name",
    columns="diagnosed_name",
    values="similarity_result"
)

# Garante que só usamos as colunas na ordem desejada e que existem no df
cols_present = [col for col in diagnosis_order if col in df_pivot.columns]
df_pivot = df_pivot[cols_present]

fig = go.Figure()

# Posições alternadas para os textos de cada candidato
positions = ["top center", "bottom center", "top left", "bottom left"]

# Scatter + linhas com texto em todas as séries, mas posições alternadas
for i, cand in enumerate(df_pivot.index):
    similarity_vals = df_pivot.loc[cand].values
    similarity_texts = [f"{v:.2f}" for v in similarity_vals]

    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=similarity_vals,
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=3),
        marker=dict(size=10),
        text=similarity_texts,
        textposition=positions[i % len(positions)],  # alterna posição do texto
        textfont=dict(size=14),
    ))

# Índice do baseline na ordem efetivamente usada
baseline_name = "diagnosed_baseline"
if baseline_name in df_pivot.columns:
    baseline_idx = df_pivot.columns.tolist().index(baseline_name)

    # Linhas verticais tracejadas finas ao redor do baseline
    fig.add_vline(
        x=baseline_idx - 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )
    fig.add_vline(
        x=baseline_idx + 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )

fig.update_layout(
    title='Similaridade dos candidatos sob erros paramétricos (±25%) no diagnóstico',
    yaxis=dict(
        title='Similarity',
        range=[0.4, 1],   # limite mínimo 0.4 e máximo 1
        dtick=0.05
    ),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=df_pivot.columns.tolist(),
        tickvals=df_pivot.columns.tolist(),
        ticktext=error_labels[:len(df_pivot.columns)]
    ),
    width=1000,
    height=700,
    legend=dict(
        title="Candidatos",
        orientation="h",
        xanchor="center", x=0.5,
        yanchor="top",    y=-0.20,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=11)
    ),
    margin=dict(l=40, r=40, t=70, b=160),
    font=dict(
        family="Arial",
        size=12
    )
)

fig.show()


In [132]:
import pandas as pd
import plotly.graph_objects as go

# Ordem dos diagnósticos no eixo X:
diagnosis_order = [
    "diagnosed_minus_100",
    "diagnosed_minus_75",
    "diagnosed_minus_50",
    "diagnosed_minus_25",
    "diagnosed_baseline",
    "diagnosed_plus_25",
    "diagnosed_plus_50",
    "diagnosed_plus_75",
    "diagnosed_plus_100",
]

# Rótulos bonitos para o eixo X (erro percentual)
error_labels = [
    "-100%", "-75%", "-50%", "-25%",
    "0%",
    "+25%", "+50%", "+75%", "+100%",
]

# Pivot: linhas = candidatos, colunas = diagnósticos, valores = similaridade
df_pivot = df.pivot(
    index="candidate_name",
    columns="diagnosed_name",
    values="similarity_result"
)

# Garante que só usamos as colunas na ordem desejada e que existem no df
cols_present = [col for col in diagnosis_order if col in df_pivot.columns]
df_pivot = df_pivot[cols_present]

fig = go.Figure()

# Posições alternadas para os textos de cada candidato (pra reduzir sobreposição)
positions = ["top center", "bottom center", "top left", "bottom left"]

# Controle de valores já rotulados por coluna (x)
# chave: nome da coluna, valor: conjunto de valores de similaridade já usados em label
seen_values = {col: set() for col in df_pivot.columns}

for i, cand in enumerate(df_pivot.index):
    similarity_vals = df_pivot.loc[cand].values

    texts_for_trace = []
    for col, val in zip(df_pivot.columns, similarity_vals):
        val_rounded = round(val, 2)  # mesma precisão do label

        # se esse valor já foi usado nessa coluna, não mostra label
        if val_rounded in seen_values[col]:
            texts_for_trace.append("")  # texto vazio -> sem rótulo
        else:
            texts_for_trace.append(f"{val_rounded:.2f}")
            seen_values[col].add(val_rounded)

    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=similarity_vals,
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=3),
        marker=dict(size=10),
        text=texts_for_trace,
        textposition=positions[i % len(positions)],
        textfont=dict(size=14),
    ))

# Índice do baseline na ordem efetivamente usada
baseline_name = "diagnosed_baseline"
if baseline_name in df_pivot.columns:
    baseline_idx = df_pivot.columns.tolist().index(baseline_name)

    fig.add_vline(
        x=baseline_idx - 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )
    fig.add_vline(
        x=baseline_idx + 0.48,
        line_width=1,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        layer='above'
    )

fig.update_layout(
    title='Similaridade dos candidatos sob erros paramétricos (±25%) no diagnóstico',
    yaxis=dict(
        title='Similarity',
        range=[0.4, 1],
        dtick=0.05
    ),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=df_pivot.columns.tolist(),
        tickvals=df_pivot.columns.tolist(),
        ticktext=error_labels[:len(df_pivot.columns)]
    ),
    width=1000,
    height=700,
    legend=dict(
        title="Candidatos",
        orientation="h",
        xanchor="center", x=0.5,
        yanchor="top",    y=-0.20,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=11)
    ),
    margin=dict(l=40, r=40, t=70, b=160),
    font=dict(
        family="Arial",
        size=12
    )
)

fig.show()
